# ② 저조도 개선 — arm 육안 검토

`docs/lowlight_classical.md` 6-2 · 6-3 의 판정을 **직접 눈으로 확인**하기 위한 노트북.
수치표만으로는 놓치는 것이 있어서 만들었다 — 실제로 6-3-4 에서 **노이즈 지표가 육안과 반대 결론**을 냈다.

## 보게 될 것

| 절 | 내용 | 답하는 질문 |
|----|------|-------------|
| 1 | 야간 표본 선택 | 무엇을 보고 판단하는가 |
| 2 | **톤커브 실측** | *왜* A1/A2 는 강광원을 못 누르는가 |
| 3 | 전체 비교 그리드 | 전반적 인상 |
| 4 | **강광원 확대** | 눈부심이 실제로 어떻게 되는가 |
| 5 | **암부 확대** | 노이즈가 얼마나 드러나는가 (지표가 놓친 것) |
| 6 | 목적 축 수치 | 표와 그림이 일치하는가 |
| 7 | 속도 | 게이트를 넘는가 |

> ⚠️ **작업 디렉토리를 `notebooks/` 로 열 것.** 데이터 경로를 `../data` 로 잡는다.
> 커널은 `.venv` (Python 3.13) 를 선택한다.

---

## arm 이름

| arm | 구성 | 계열 |
|-----|------|------|
| `none` | 무처리 | 대조군 — 이게 없으면 ②가 도움이 되는지 자체를 모른다 |
| `A1+bf` | CLAHE + 감마 + bilateral | 타일 적응 톤커브 |
| `A2` | AGCWD | 전역 적응 감마 |
| `D1` | Tonemap Drago | **하이라이트 압축** |
| `R1` | Tonemap Reinhard | 하이라이트 압축 |
| `L1` | LIME 근사 | Retinex — 하이라이트 유지 |

In [ ]:
import sys
from pathlib import Path

import cv2
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent  # notebooks/ 의 상위 = 저장소 루트
assert (ROOT / "data").is_dir(), (
    f"data/ 를 찾을 수 없다. 작업 디렉토리를 notebooks/ 로 열 것 (현재: {Path.cwd()})"
)
sys.path.insert(0, str(ROOT / "scripts"))

from lowlight import build  # noqa: E402
from night_eval import POOLS, list_images, luma, measure  # noqa: E402

# --- 표시 설정 ---------------------------------------------------------
# 노트북 테마(라이트/다크)와 무관하게 그림 표면을 고정한다. 팔레트가 이 표면
# 기준으로 대비 검증된 값이라, 배경이 바뀌면 검증이 무효가 된다.
matplotlib.rcParams.update({
    "font.family": ["Malgun Gothic", "sans-serif"],  # 한글 라벨
    "axes.unicode_minus": False,
    "figure.facecolor": "#fcfcfb",
    "axes.facecolor": "#fcfcfb",
    "savefig.facecolor": "#fcfcfb",
    "text.color": "#0b0b0b",
    "axes.labelcolor": "#52514e",
    "axes.edgecolor": "#c3c2b7",
    "xtick.color": "#898781",
    "ytick.color": "#898781",
    "grid.color": "#e1e0d9",
    "figure.dpi": 110,
})

ARMS = ["none", "A1+bf", "A2", "D1", "R1", "L1"]

# 'none' 은 대조군이므로 계열 색을 주지 않는다(무채색).
# 나머지는 고정 슬롯 순서 — 시리즈가 빠져도 색을 재배치하지 않는다.
ARM_COLOR = {
    "none": "#898781",
    "A1+bf": "#2a78d6",
    "A2": "#eb6834",
    "D1": "#1baf7a",
    "R1": "#eda100",
    "L1": "#e87ba4",
}
INK, MUTED = "#0b0b0b", "#898781"

arms = {name: build(name) for name in ARMS}
for name, arm in arms.items():
    print(f"{name:<7} {arm.describe()}")

## 1. 야간 표본 선택

평균밝기 60 이하만 고른다. 풀별로 고르게 뽑아 한 데이터셋이 인상을 지배하지 않게 한다.

> ⚠️ **ExDark 는 야간이지만 실내·근거리 사물이 많다** (`lowlight_classical.md` 6-3-5).
> 본 프로젝트 도메인은 야외 야간 **보행**이라 장면 통계가 다르다.
> NightOwls Validation 이 도착하면 `POOLS` 에 추가해 다시 볼 것.

In [ ]:
MAX_LUMA = 60.0  # 야간 판정 상한
rng = np.random.default_rng(0)


def collect_night(pool_names=("exdark", "stair", "loli_test"), per_pool=3, scan=250):
    """풀별로 평균밝기 <= MAX_LUMA 인 영상을 고르게 뽑는다."""
    picked = []
    for name in pool_names:
        paths = list_images(POOLS[name])
        if not paths:
            print(f"  [건너뜀] {name} — 경로 없음: {POOLS[name]}")
            continue
        order = rng.permutation(len(paths))[:scan]
        taken = 0
        for i in order:
            if taken >= per_pool:
                break
            img = cv2.imread(str(paths[i]))
            if img is None or float(luma(img).mean()) > MAX_LUMA:
                continue
            picked.append((name, paths[i].name, img))
            taken += 1
    return picked


# 그림용과 통계용을 분리한다.
#   그림은 격자에 들어가야 해서 적을 수밖에 없는데, 그 표본으로 평균을 내면
#   문서(n=63)와 다른 수치가 나와 비교가 안 된다. 통계는 따로 많이 뽑는다.
samples = collect_night(per_pool=3)
imgs = [im for _, _, im in samples]

stat_samples = collect_night(per_pool=14)
stat_imgs = [im for _, _, im in stat_samples]

print(f"그림용 {len(samples)}장 · 통계용 {len(stat_imgs)}장\n")
print(f"{'pool':<11}{'파일':<30}{'해상도':>12}{'평균밝기':>10}{'포화면적':>10}")
print("-" * 73)
for pool, fname, img in samples:
    L = luma(img)
    print(f"{pool:<11}{fname[:28]:<30}{f'{img.shape[1]}x{img.shape[0]}':>12}"
          f"{L.mean():>10.1f}{100 * (L >= 250).mean():>9.2f}%")

## 2. 톤커브 실측 — 왜 A1/A2 는 강광원을 못 누르는가 ★

표본 전체의 픽셀을 **입력 휘도 → 출력 휘도**로 모아 중앙값 곡선을 그린다.
CLAHE 처럼 국소적으로 동작하는 기법도 포함해 *실제* 전달 특성이 나온다.

**읽는 법** — 회색 점선(항등선)보다 **위**면 그 밝기를 올린 것, **아래**면 내린 것이다.
오른쪽 음영이 강광원 구간(상위 1%)이며, **여기서 곡선이 항등선 아래로 내려가야 눈부심을 억제한 것**이다.

In [ ]:
def tone_curve(arm, images, bins=52, min_px=200):
    """입력 휘도 -> 출력 휘도 실측 전달함수.

    영상마다 곡선을 따로 구한 뒤 영상 간 중앙값을 취한다. 전체 픽셀을 한 번에
    모으면 큰 영상·밝은 영상이 곡선을 지배하고, 구간 표본이 들쭉날쭉해져
    곡선이 튄다. min_px 미만인 구간은 버린다.
    """
    edges = np.linspace(0, 255, bins + 1)
    centers = (edges[:-1] + edges[1:]) / 2

    per_image = []
    for img in images:
        x, y = luma(img).ravel(), luma(arm(img)).ravel()
        slot = np.clip(np.digitize(x, edges) - 1, 0, bins - 1)
        med = np.full(bins, np.nan)
        for b in range(bins):
            m = slot == b
            if m.sum() >= min_px:
                med[b] = np.median(y[m])
        per_image.append(med)

    stack = np.vstack(per_image)
    enough = np.isfinite(stack).sum(axis=0) >= 3  # 영상 3장 이상이 채운 구간만
    out = np.full(bins, np.nan)
    if enough.any():
        out[enough] = np.nanmedian(stack[:, enough], axis=0)
    return centers, out


# 밝은 영역의 경계를 **고정값**으로 둔다.
# 상위 1% 같은 분위수로 잡으면 포화가 심한 표본에서 경계가 253 까지 올라가
# 판정 구간이 두어 칸으로 쪼그라든다(실제로 8-2 에서 그렇게 됐다).
GLARE_MIN = 200.0

fig, ax = plt.subplots(figsize=(8.8, 6.2))

# 'none' 은 정의상 항등선 그 자체라 곡선으로 그리지 않는다 (회색 점선이 곧 none)
CURVE_ARMS = [n for n in ARMS if n != "none"]
curves = {name: tone_curve(arms[name], stat_imgs) for name in CURVE_ARMS}

ax.axvspan(GLARE_MIN, 255, color="#eda100", alpha=0.10, zorder=0)
ax.text(GLARE_MIN + 3, 10, f"밝은 영역\n(>{GLARE_MIN:.0f})",
        fontsize=9, color="#52514e", va="bottom")

ax.plot([0, 255], [0, 255], color="#898781", lw=1.4, ls=(0, (5, 4)), zorder=1,
        label="항등선 = 무처리(none)")

for name in CURVE_ARMS:
    cx, cy = curves[name]
    ax.plot(cx, cy, color=ARM_COLOR[name], lw=2, zorder=3, label=name,
            solid_capstyle="round")

# 직접 라벨 — 결론을 좌우하는 둘만. 나머지는 범례가 담당한다
for name, dy in (("D1", -18), ("A2", 10)):
    cx, cy = curves[name]
    ok = np.isfinite(cy)
    ax.annotate(name, (cx[ok][-1], cy[ok][-1] + dy), color=ARM_COLOR[name],
                fontsize=11, fontweight="bold", ha="right")

ax.set_xlim(0, 258)
ax.set_ylim(0, 276)
ax.set_xlabel("입력 휘도")
ax.set_ylabel("출력 휘도 (구간 중앙값)")
ax.set_title("톤커브 실측 — 밝은 영역에서 항등선 아래로 내려가는 arm 만이 눈부심을 억제한다",
             fontsize=11, color=INK, pad=14)
ax.grid(True, lw=0.6, alpha=0.7)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(loc="upper left", frameon=False, fontsize=9, labelcolor="#52514e")
plt.tight_layout()
plt.show()

# 표 병기 — 색 대비 WARN 에 대한 relief(가시 라벨 + 표)
print(f"\n밝은 영역(>{GLARE_MIN:.0f}) 평균 출력 휘도")
print(f"{'arm':<8}{'출력':>8}{'항등 대비':>12}   판정")
print("-" * 42)
for name in CURVE_ARMS:
    cx, cy = curves[name]
    m = (cx >= GLARE_MIN) & np.isfinite(cy)
    if not m.any():
        continue
    out, ident = cy[m].mean(), cx[m].mean()
    print(f"{name:<8}{out:>8.1f}{out - ident:>+12.1f}   "
          f"{'억제 O' if out < ident else '증폭 X'}")

## 3. 전체 비교 그리드

행 = 표본, 열 = arm. 전반적 인상을 본다.

In [ ]:
def show_row_grid(items, arm_names, col_w=2.7, title=None):
    """행=표본, 열=[원본 + arm들] 격자."""
    cols = 1 + len(arm_names)
    rows = len(items)
    ar = items[0][2].shape[0] / items[0][2].shape[1]
    fig, axes = plt.subplots(rows, cols, figsize=(col_w * cols, col_w * ar * rows))
    axes = np.atleast_2d(axes)

    for r, (pool, fname, img) in enumerate(items):
        cells = [("원본", img)] + [(n, arms[n](img)) for n in arm_names]
        for c, (label, out) in enumerate(cells):
            ax = axes[r, c]
            ax.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
            ax.set_xticks([])
            ax.set_yticks([])
            for s in ax.spines.values():
                s.set_color("#e1e0d9")
            if r == 0:
                color = ARM_COLOR.get(label, INK)
                ax.set_title(label, fontsize=10, color=color, fontweight="bold", pad=6)
        axes[r, 0].set_ylabel(f"{pool}", fontsize=8, color=MUTED)

    if title:
        fig.suptitle(title, fontsize=11, color=INK, y=1.0)
    plt.tight_layout()
    plt.show()


show_row_grid(samples, ARMS, title="arm 별 전체 결과")

## 4. 강광원 확대 — 눈부심 축 ★

가장 밝은 **덩어리**(단일 픽셀이 아니라 광원) 주변을 잘라 확대한다.
가로등·전조등이 어떻게 처리되는지가 이 프로젝트의 핵심이다 — 대상 사용자가 **광과민** 야맹증이기 때문이다.

In [ ]:
def crop_around(img, cy, cx, size):
    h, w = img.shape[:2]
    half = size // 2
    y0 = int(np.clip(cy - half, 0, max(0, h - size)))
    x0 = int(np.clip(cx - half, 0, max(0, w - size)))
    return img[y0:y0 + size, x0:x0 + size]


def brightest_spot(img, blur=9):
    """광원 '덩어리' 중심. 블러를 먹여 단일 핫픽셀에 낚이지 않게 한다."""
    L = cv2.GaussianBlur(luma(img), (0, 0), blur)
    _, _, _, (mx, my) = cv2.minMaxLoc(L)
    return my, mx


def show_crops(items, arm_names, locator, size=190, col_w=2.4, title=None, note=None):
    cols = 1 + len(arm_names)
    fig, axes = plt.subplots(len(items), cols, figsize=(col_w * cols, col_w * len(items)))
    axes = np.atleast_2d(axes)

    for r, (pool, fname, img) in enumerate(items):
        cy, cx = locator(img)
        s = min(size, img.shape[0], img.shape[1])
        cells = [("원본", crop_around(img, cy, cx, s))]
        cells += [(n, crop_around(arms[n](img), cy, cx, s)) for n in arm_names]
        for c, (label, patch) in enumerate(cells):
            ax = axes[r, c]
            ax.imshow(cv2.cvtColor(patch, cv2.COLOR_BGR2RGB), interpolation="nearest")
            ax.set_xticks([])
            ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_color("#e1e0d9")
            if r == 0:
                ax.set_title(label, fontsize=10, fontweight="bold",
                             color=ARM_COLOR.get(label, INK), pad=6)
        axes[r, 0].set_ylabel(pool, fontsize=8, color=MUTED)

    if title:
        fig.suptitle(title, fontsize=11, color=INK, y=1.0)
    plt.tight_layout()
    plt.show()
    if note:
        print(note)


show_crops(samples, ARMS, brightest_spot,
           title="강광원 주변 확대 — halo 가 커지는가, 눌리는가",
           note="흰색으로 뭉개진 면적이 늘었다면 그 영역의 정보는 소실된 것이다.")

## 5. 암부 확대 — 노이즈 ★ (지표가 놓친 것)

어둡고 **평탄한** 영역을 잘라 확대한다. 구조가 없는 곳이라 보이는 얼룩은 전부 노이즈다.

> ⚠️ `lowlight_classical.md` 6-3-4 — **현재 노이즈 지표는 여기서 보이는 것과 반대 결론을 낸다.**
> 지표는 전역 밝기이득으로 정규화하는데, 톤매핑 계열은 암부를 크게 끌어올려
> *원래 안 보이던 노이즈를 드러내면서* 밝기이득도 함께 키우기 때문이다.
> **이 그림이 지표보다 신뢰할 만하다.**

In [ ]:
def darkest_flat_spot(img, size=190):
    """어두우면서 국소 대비가 낮은 곳 = 구조가 없어 노이즈만 남는 영역."""
    L = luma(img)
    k = (25, 25)
    mean = cv2.blur(L, k)
    var = cv2.blur(L * L, k) - mean * mean
    # 완전 검정(정보 0)은 제외 — 볼 것이 없다
    score = np.where(mean < 4, np.inf, mean + 3.0 * np.sqrt(np.maximum(var, 0)))
    half = size // 2
    inner = score[half:-half or None, half:-half or None]
    if inner.size == 0:
        return L.shape[0] // 2, L.shape[1] // 2
    y, x = np.unravel_index(np.argmin(inner), inner.shape)
    return y + half, x + half


show_crops(samples, ARMS, darkest_flat_spot,
           title="암부 평탄영역 확대 — 보이는 얼룩은 전부 노이즈다",
           note="컬러 얼룩이 심할수록 탐지기 입력으로도, 표시용으로도 불리하다.")

## 6. 목적 축 수치 — 그림과 표가 일치하는가

`scripts/night_eval.py` 와 **같은 함수**로 계산한다. 표본이 작으니 절대값이 아니라 **순위**만 본다.

In [ ]:
stats = {name: measure(arms[name], stat_imgs) for name in ARMS}

print(f"통계 표본 {len(stat_imgs)}장\n")
print(f"{'arm':<8}{'강광원 변화':>13}{'포화면적 전→후':>20}{'정규화대비':>12}{'노이즈증폭(지표)':>18}")
print("-" * 73)
for name in ARMS:
    s = stats[name]
    sat = f"{s['sat_before']:.2f}% → {s['sat_after']:.2f}%"
    print(f"{name:<8}{s['glare_after'] - s['glare_before']:>+13.1f}{sat:>20}"
          f"{s['norm_contrast']:>12.2f}{s['noise_amp']:>18.3f}")

print("\n⚠️ 노이즈증폭 지표는 6-3-4 기준 신뢰할 수 없다 — 5절 그림과 대조할 것.")
print("   표본이 문서(n=63)보다 작아 절대값은 다를 수 있다. 순위를 볼 것.")

In [ ]:
series = [n for n in ARMS if n != "none"]  # 대조군은 정의상 0/1 이라 막대로 그리지 않는다
colors = [ARM_COLOR[n] for n in series]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.4))

# --- 왼쪽: 강광원 변화 (음수여야 목적 달성) ---
vals = [stats[n]["glare_after"] - stats[n]["glare_before"] for n in series]
ax1.axhline(0, color="#c3c2b7", lw=1.2, zorder=1)
bars = ax1.bar(series, vals, color=colors, width=0.62, zorder=2)
for b, v in zip(bars, vals):
    ax1.text(b.get_x() + b.get_width() / 2, v + (3 if v >= 0 else -3), f"{v:+.0f}",
             ha="center", va="bottom" if v >= 0 else "top",
             fontsize=9, color="#52514e", fontweight="bold")
ax1.set_ylabel("상위 1% 휘도 변화")
ax1.set_title("눈부심 — 아래로 내려가야 억제한 것", fontsize=10.5, color=INK, pad=10)

# --- 오른쪽: 정규화 대비 (1.0 넘어야 '대비 강조') ---
vals2 = [stats[n]["norm_contrast"] for n in series]
top = max(vals2) * 1.30
ax2.axhline(1.0, color="#c3c2b7", lw=1.2, ls=(0, (5, 4)), zorder=1)
# 기준선 설명은 막대 위쪽 빈 공간에 둔다 — 선 바로 옆에 두면 1.0 근처 막대의
# 값 라벨과 겹친다 (R1 이 1.02 라 실제로 겹쳤다)
ax2.text(-0.45, top * 0.955, "점선 1.0 = 밝히기와 대비강조의 경계",
         fontsize=8.5, color="#52514e", ha="left", va="top")
bars2 = ax2.bar(series, vals2, color=colors, width=0.62, zorder=2)
for b, v in zip(bars2, vals2):
    ax2.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.2f}",
             ha="center", fontsize=9, color="#52514e", fontweight="bold")
ax2.set_ylabel("정규화 대비이득")
ax2.set_ylim(0, top)
ax2.set_title("대비 — 밝기가 아니라 대비를 올렸는가", fontsize=10.5, color=INK, pad=10)

for ax in (ax1, ax2):
    ax.grid(True, axis="y", lw=0.6, alpha=0.7)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

plt.tight_layout()
plt.show()

## 7. 속도 — 1차 게이트

`② 단독 ≤ 20ms/frame @720p` (`data.md` 2-3-3). CPU·NumPy 측정이라 **폰 성능이 아니다** — 순위 참고용.

In [ ]:
import time

GATE_MS = 20.0
SIZES = [(1280, 720), (854, 480), (640, 360)]


def bench(arm, frame, runs=11):
    """runs 를 넉넉히 둔다 — A1+bf 가 게이트 바로 근처라 분산이 결론을 뒤집는다."""
    for _ in range(3):
        arm(frame)
    t = []
    for _ in range(runs):
        t0 = time.perf_counter()
        arm(frame)
        t.append((time.perf_counter() - t0) * 1000)
    return float(np.median(t))


base = imgs[0]
speed = {
    n: [bench(arms[n], cv2.resize(base, s, interpolation=cv2.INTER_CUBIC)) for s in SIZES]
    for n in series
}

fig, ax = plt.subplots(figsize=(9.2, 4.6))
x = np.arange(len(SIZES))
w = 0.16
for i, name in enumerate(series):
    off = (i - (len(series) - 1) / 2) * w
    b = ax.bar(x + off, speed[name], width=w - 0.02, color=ARM_COLOR[name],
               label=name, zorder=2)
    for rect, v in zip(b, speed[name]):
        ax.text(rect.get_x() + rect.get_width() / 2, v + 1.5, f"{v:.0f}",
                ha="center", fontsize=7.5, color="#52514e")

ax.axhline(GATE_MS, color="#d03b3b", lw=1.4, ls=(0, (5, 4)), zorder=3)
ax.text(len(SIZES) - 0.5, GATE_MS + 2.5, f"1차 게이트 {GATE_MS:.0f}ms",
        color="#d03b3b", fontsize=9, ha="right", fontweight="bold")

ax.set_xticks(x, [f"{w_}x{h_}" for w_, h_ in SIZES])
ax.set_ylabel("ms / frame (중앙값)")
ax.set_title("내부 처리 해상도별 비용 — 해상도를 낮추면 후보군 자체가 넓어진다",
             fontsize=10.5, color=INK, pad=12)
ax.grid(True, axis="y", lw=0.6, alpha=0.7)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, fontsize=9, ncols=len(series), labelcolor="#52514e",
          loc="upper right")
plt.tight_layout()
plt.show()

print(f"{'arm':<8}" + "".join(f"{f'{w_}x{h_}':>12}" for w_, h_ in SIZES) + f"{'720p 게이트':>14}")
print("-" * 60)
for name in series:
    row = "".join(f"{v:>11.1f}ms" for v in speed[name])
    ms720 = speed[name][0]
    verdict = "OK" if ms720 <= GATE_MS else "탈락"
    if abs(ms720 - GATE_MS) <= 4:
        verdict += " (경계)"
    print(f"{name:<8}{row}{verdict:>14}")

print("\n한 장 기준이라 장면에 따라 흔들린다. A1+bf 는 게이트 바로 근처라 실행마다 판정이 바뀔 수 있다.")
print("CPU·NumPy 측정이며 폰 성능이 아니다 — 기각 근거로 쓰지 말 것.")

---

# 8. 고전 기법 **전수** 적용 — 5개 계열 지형도 ★

앞 절까지는 **후보 arm 5개**(A1+bf · A2 · D1 · R1 · L1)만 봤다.
그런데 `lowlight_classical.md` 1장의 지형도에는 **12개 기법 / 5개 계열**이 있고,
그중 상당수는 **2장의 CPU 속도 프로브에서 탈락시킨 뒤 한 번도 눈으로 본 적이 없다.**

속도로만 거른 것은 위험하다 — 2장 스스로 *"CPU·NumPy 측정이지 폰 성능이 아니다,
기각 근거로 쓰지 말 것"* 이라고 적어두었다. 여기서 **실제 야간 이미지에 전부 적용해 본다.**

| 계열 | 기법 | 지금까지의 취급 |
|------|------|------------------|
| ① 히스토그램·톤커브 | 감마 · **HE** · CLAHE · AGCWD | CLAHE/AGCWD 만 arm 채택, HE 는 미검토 |
| ② Retinex | **SSR** · **MSR** · LIME | SSR/MSR 은 속도로 탈락, **육안 미확인** |
| ③ 다중노출 융합 | **Mertens** | 속도로 탈락, **육안 미확인** |
| ④ Dehaze 역변환 | **Dong 2011** | 속도로 탈락, **육안 미확인** |
| ⑤ HDR 톤매핑 | Reinhard · Drago · **Mantiuk** | Mantiuk 만 미확인 |

> 이 기법들은 **후보 arm 이 아니라 지형도 확인용**이다. 학습 파이프라인에서 재사용하지
> 않으므로 `scripts/lowlight.py` 의 arm 레지스트리를 오염시키지 않고 이 노트북에서만 정의한다.

In [ ]:
from lowlight import CLAHE, AGCWD, LIME, guided_filter  # noqa: E402


def hdr_tonemap(bgr, tm):
    # sRGB 를 선형화해 HDR 오퍼레이터에 넣고 표시 감마를 되씌운다
    lin = np.power(bgr.astype(np.float32) / 255.0, 2.2)
    out = np.nan_to_num(tm.process(lin), nan=0.0, posinf=1.0, neginf=0.0)
    return np.clip(out * 255.0, 0, 255).astype(np.uint8)


# --- 1 히스토그램·톤커브 -------------------------------------------------
def m_gamma(bgr, g=0.45):
    lut = np.clip(((np.arange(256) / 255.0) ** g) * 255, 0, 255).astype(np.uint8)
    return cv2.LUT(bgr, lut)


def m_he(bgr):
    # 전역 히스토그램 평활. CLAHE 에서 타일·clip 을 뺀 원형(原型)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    lab[:, :, 0] = cv2.equalizeHist(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


# --- 2 Retinex -----------------------------------------------------------
def _retinex(bgr, sigmas):
    # log(I) - log(blur(I)) = 반사율 추정. 스케일이 클수록 전역 조도를 본다
    src = bgr.astype(np.float32)
    log_i = np.log1p(src)
    r = np.zeros_like(src)
    for s in sigmas:
        r += (log_i - np.log1p(cv2.GaussianBlur(src, (0, 0), s))) / len(sigmas)
    return cv2.normalize(r, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)


def m_ssr(bgr):
    return _retinex(bgr, (80,))


def m_msr(bgr):
    return _retinex(bgr, (15, 80, 250))


# --- 3 다중노출 융합 ------------------------------------------------------
_merge = cv2.createMergeMertens()


def m_mertens(bgr):
    # 단일 프레임에서 가상 노출 스택을 만들어 피라미드 융합
    #
    # ⚠️ MergeMertens 는 **8bit 입력이어야 한다.** float32 [0,1] 을 넣으면 출력이
    #    0 근처로 뭉개져 새까맣게 나온다(실측: max 0.0057). 속도만 재던 단계에서는
    #    드러나지 않아 scripts/probe_classical.py 도 같은 버그를 갖고 있었다.
    src = bgr.astype(np.float32) / 255.0
    stack = [(np.clip(src * k, 0, 1) * 255).astype(np.uint8) for k in (1.0, 2.5, 5.0)]
    return (np.clip(_merge.process(stack), 0, 1) * 255).astype(np.uint8)


# --- 4 Dehaze 역변환 ------------------------------------------------------
def m_dong(bgr):
    # 저조도 영상을 반전하면 안개 영상과 통계가 닮는다 -> dehaze -> 재반전
    src = bgr.astype(np.float32) / 255.0
    inv = 1.0 - src
    dark = cv2.erode(inv.min(axis=2), np.ones((15, 15), np.uint8))
    t = guided_filter(inv.mean(axis=2), 1.0 - 0.9 * dark, 20, 1e-3, scale=4)
    t = np.maximum(t, 0.1)
    out = 1.0 - np.clip((inv - 1.0) / t[:, :, None] + 1.0, 0, 1)
    return (out * 255).astype(np.uint8)


# --- 5 HDR 톤매핑 ---------------------------------------------------------
_tm_reinhard = cv2.createTonemapReinhard(2.2, 0.0, 0.8, 0.0)
_tm_drago = cv2.createTonemapDrago(2.2, 1.0, 0.85)
_tm_mantiuk = cv2.createTonemapMantiuk(2.2, 0.85, 1.0)

_clahe, _agcwd, _lime = CLAHE(), AGCWD(), LIME()

# 계열 번호는 lowlight_classical.md 1장 지형도와 대응한다
FAMILIES = {
    "① 히스토그램·톤커브": [("감마 0.45", m_gamma), ("HE", m_he),
                            ("CLAHE", _clahe), ("AGCWD", _agcwd)],
    "② Retinex": [("SSR", m_ssr), ("MSR", m_msr), ("LIME", _lime)],
    "③ 다중노출 융합": [("Mertens", m_mertens)],
    "④ Dehaze 역변환": [("Dong 2011", m_dong)],
    "⑤ HDR 톤매핑": [("Reinhard", lambda b: hdr_tonemap(b, _tm_reinhard)),
                     ("Drago", lambda b: hdr_tonemap(b, _tm_drago)),
                     ("Mantiuk", lambda b: hdr_tonemap(b, _tm_mantiuk))],
}
FAMILY_COLOR = {
    "① 히스토그램·톤커브": "#2a78d6",
    "② Retinex": "#eb6834",
    "③ 다중노출 융합": "#1baf7a",
    "④ Dehaze 역변환": "#eda100",
    "⑤ HDR 톤매핑": "#e87ba4",
}
ALL_METHODS = [(fam, n, f) for fam, its in FAMILIES.items() for n, f in its]


def fit(img, max_w=640):
    # MSR 은 sigma 250 가우시안이라 큰 영상에서 초 단위로 느려진다. 폭을 제한한다
    if img.shape[1] <= max_w:
        return img
    s = max_w / img.shape[1]
    return cv2.resize(img, (max_w, int(img.shape[0] * s)), interpolation=cv2.INTER_AREA)


# 결과가 통째로 깨지면 표에서는 '억제'처럼 보인다 — 출력 밝기로 먼저 걸러낸다
print("동작 점검 (결과가 새까맣거나 새하얗지 않은가)")
probe = fit(samples[0][2])
for fam, name, fn in ALL_METHODS:
    out = fn(probe)
    mean = float(luma(out).mean())
    flag = "  <-- 이상" if mean < 5 or mean > 250 else ""
    print(f"  {name:<12} 출력 평균밝기 {mean:6.1f}{flag}")

# 강광원이 실제로 들어 있는 장면을 고른다 — 글레어 거동이 이 절의 핵심이다
survey = sorted(samples, key=lambda s: -float((luma(s[2]) >= 250).mean()))[:3]
survey = [(pool, name, fit(img)) for pool, name, img in survey]

print(f"\n기법 {len(ALL_METHODS)}개 / 계열 {len(FAMILIES)}개")
print(f"적용 대상 {len(survey)}장 (포화 광원이 많은 순)\n")
for pool, name, img in survey:
    L = luma(img)
    print(f"  {pool:<10} {name[:26]:<28} {img.shape[1]}x{img.shape[0]}"
          f"  평균 {L.mean():5.1f}  포화 {100 * (L >= 250).mean():.2f}%")

### 8-1. 계열별 결과

계열마다 그림을 나눈다 — 12개를 한 줄에 늘어놓으면 폭이 넓어 비교가 안 된다.
**계열 안에서 먼저 비교하고, 계열끼리는 8-2 의 톤커브로 비교한다.**

행은 **포화 광원이 많은 순**으로 고른 야간 장면이다 (글레어 거동을 보려면 광원이 있어야 한다).

In [ ]:
def show_family(fam, items, images, col_w=2.9):
    cols = 1 + len(items)
    fig, axes = plt.subplots(len(images), cols,
                             figsize=(col_w * cols, col_w * 0.8 * len(images)))
    axes = np.atleast_2d(axes)
    if cols == 1:
        axes = axes.reshape(len(images), 1)

    for r, (pool, name, img) in enumerate(images):
        cells = [("원본", img)] + [(n, fn(img)) for n, fn in items]
        for c, (label, out) in enumerate(cells):
            ax = axes[r, c]
            ax.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
            ax.set_xticks([])
            ax.set_yticks([])
            for sp in ax.spines.values():
                sp.set_color("#e1e0d9")
            if r == 0:
                ax.set_title(label, fontsize=10, fontweight="bold", pad=6,
                             color=INK if label == "원본" else FAMILY_COLOR[fam])
        axes[r, 0].set_ylabel(pool, fontsize=8, color=MUTED)

    fig.suptitle(fam, fontsize=12, color=FAMILY_COLOR[fam], fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()


for fam, items in FAMILIES.items():
    show_family(fam, items, survey)

### 8-2. 계열별 톤커브 — 어느 계열이 구조적으로 강광원을 누르는가

2절을 12개 기법 전부로 확장한다. **색 = 계열, 선종류 = 계열 안의 기법**이라
같은 계열끼리 뭉치는지(= 계열 특성이 실재하는지) 바로 보인다.

In [ ]:
curve_imgs = [fit(im) for _, _, im in samples]  # 곡선은 표본이 많아야 안정적
fam_curves = {name: tone_curve(fn, curve_imgs, min_px=80)
              for _, name, fn in ALL_METHODS}
DASHES = [(0, ()), (0, (6, 3)), (0, (2, 2)), (0, (8, 3, 2, 3))]

fig, ax = plt.subplots(figsize=(9.4, 6.8))

ax.axvspan(GLARE_MIN, 255, color="#eda100", alpha=0.10, zorder=0)
ax.text(GLARE_MIN + 3, 8, f"밝은 영역\n(>{GLARE_MIN:.0f})", fontsize=9,
        color="#52514e", va="bottom")
ax.plot([0, 255], [0, 255], color="#898781", lw=1.4, ls=(0, (5, 4)), zorder=1,
        label="항등선 = 무처리")

# 색 = 계열(정체성), 선종류 = 계열 안의 기법 — 복합 부호화
for fam, items in FAMILIES.items():
    for k, (name, _) in enumerate(items):
        cx, cy = fam_curves[name]
        ax.plot(cx, cy, color=FAMILY_COLOR[fam], lw=1.9, ls=DASHES[k % len(DASHES)],
                zorder=3, label=f"{fam[0]} {name}")

ax.set_xlim(0, 258)
ax.set_ylim(0, 268)
ax.set_xlabel("입력 휘도")
ax.set_ylabel("출력 휘도 (구간 중앙값)")
ax.set_title("계열별 톤커브 — 밝은 영역에서 항등선 아래로 내려가는 계열은 어디인가",
             fontsize=11, color=INK, pad=14)
ax.grid(True, lw=0.6, alpha=0.7)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
# 13개 항목이라 축 안에 두면 곡선을 덮는다 — 아래로 뺀다
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), frameon=False,
          fontsize=8.5, ncols=5, labelcolor="#52514e")
plt.tight_layout()
plt.show()

print(f"밝은 영역(>{GLARE_MIN:.0f}) 평균 출력 휘도 — 항등 대비")
print(f"{'계열':<18}{'기법':<12}{'출력':>8}{'항등대비':>10}   판정")
print("-" * 62)
last = None
for fam, name, _ in ALL_METHODS:
    cx, cy = fam_curves[name]
    m = (cx >= GLARE_MIN) & np.isfinite(cy)
    if not m.any():
        continue
    out, ident = cy[m].mean(), cx[m].mean()
    tag = fam if fam != last else ""
    last = fam
    print(f"{tag:<18}{name:<12}{out:>8.1f}{out - ident:>+10.1f}   "
          f"{'억제 O' if out < ident else '증폭 X'}")

### 8-3. 전 기법 요약 — 속도 · 글레어 · 대비

`night_eval.py` 와 같은 `measure()` 로 계산한다. **속도는 480p 기준**이다
(720p 로 재면 MSR·Mantiuk 이 초 단위라 노트북이 멈춘 것처럼 보인다).

In [ ]:
summary = []
speed_frame = cv2.resize(survey[0][2], (854, 480), interpolation=cv2.INTER_CUBIC)
for fam, name, fn in ALL_METHODS:
    ms = bench(fn, speed_frame, runs=5)
    st = measure(fn, curve_imgs)
    summary.append((fam, name, ms, st["glare_after"] - st["glare_before"],
                    st["sat_after"] - st["sat_before"], st["norm_contrast"]))

print(f"{'계열':<18}{'기법':<12}{'480p ms':>10}{'강광원':>9}{'포화Δ':>9}{'정규화대비':>12}  글레어")
print("-" * 84)
last = None
for fam, name, ms, dg, dsat, nc in summary:
    tag = fam if fam != last else ""
    last = fam
    print(f"{tag:<18}{name:<12}{ms:>10.1f}{dg:>+9.1f}{dsat:>+8.2f}%{nc:>12.2f}"
          f"  {'억제 O' if dg < 0 else '증폭 X'}")

print("\n480p 기준이다. 720p 는 대략 2.3배로 잡으면 된다 (게이트 20ms @720p).")
print("표본 소수 기준이라 절대값이 아니라 계열 간 경향만 볼 것.")

---

## 정리 — 확인해야 할 것

문서(`docs/lowlight_classical.md`)의 결론이 **위 그림과 맞는지** 직접 판단할 대목:

### 후보 arm (2~7절)

| # | 확인 사항 | 어디서 |
|---|-----------|--------|
| 1 | A1/A2 곡선이 밝은 영역에서 항등선 **위**에 있는가 (= 억제 불가) | 2절 |
| 2 | D1 만 항등선 **아래**로 내려가는가 | 2절 · 4절 |
| 3 | A2 에서 흰 뭉개짐 면적이 늘어나는가 | 4절 |
| 4 | **D1/R1/L1 의 암부 노이즈가 A1+bf 보다 심한가** — 지표는 반대로 말한다 | 5절 vs 6절 |
| 5 | D1 이 글레어를 누른 대가로 전체가 어두워지지 않았는가 | 3절 |
| 6 | 색 왜곡: A1/A2 는 *탈색*, D1/R1/L1 은 *캐스트* — 다른 문제인가 | 3절 |

**4번이 가장 중요하다.** 눈으로 본 것과 지표가 어긋나면 **지표를 고쳐야지 눈을 의심하면 안 된다.**

### 전수 지형도 (8절)

| # | 확인 사항 | 어디서 |
|---|-----------|--------|
| 7 | **SSR·MSR 은 색이 통째로 빠져 뿌옇게 된다** — 야간 조명의 색 정보가 사라진다. 속도(480p 수백~2000ms)보다 이쪽이 더 결정적인 탈락 사유 아닌가 | 8-1 ② |
| 8 | **HE 는 CLAHE 보다 과하게 밝히고 강광원을 더 키운다**(+101 vs +45). CLAHE 의 clip·타일이 실제로 일을 하는가 | 8-1 ① · 8-3 |
| 9 | **⑤ HDR 톤매핑만 밝은 영역에서 항등선 아래로 내려간다** — 계열 특성이 실재하는가 | 8-2 |
| 10 | Mertens·Dong 은 육안으로 쓸 만한가, 아니면 속도 이전에 화질로 탈락인가 | 8-1 ③④ |

> ⚠️ **8절에서 버그를 하나 잡았다** — `MergeMertens` 는 **8bit 입력이어야 한다.**
> float32 [0,1] 을 넣으면 출력이 새까맣게 나오는데(실측 max 0.0057), 속도만 재던
> `scripts/probe_classical.py` 는 이걸 모른 채 통과시켰다. 고치고 나니 Mertens 판정이
> **"억제 −147" → "증폭 +86"** 으로 뒤집혔다. **결과를 눈으로 보지 않으면 지표가 거짓말을 한다**는
> 사례라 8절 첫 셀에 출력 밝기 점검을 넣어두었다.

### 표본을 바꿔보려면

```python
rng = np.random.default_rng(7)        # 다른 표본
samples = collect_night(per_pool=5)   # 더 많이
imgs = [im for _, _, im in samples]
```

이후 2절부터 다시 실행. NightOwls 가 도착하면 `night_eval.POOLS` 에 등록한 뒤
`collect_night(pool_names=("nightowls", ...))` 로 **야외 보행 시점**에서 재확인할 것 —
현재 ExDark 표본은 실내가 섞여 있어 도메인이 완전히 맞지 않는다 (`lowlight_classical.md` 6-3-5).